In [1]:
# Processing csv files from the GBD for country specific BMR to xarray format
# GBD Results tool:
# Use the following to cite data included in this download:
# Global Burden of Disease Collaborative Network.
# Global Burden of Disease Study 2021 (GBD 2021) Results.
# Seattle, United States: Institute for Health Metrics and Evaluation (IHME), 2022.
# Available from https://vizhub.healthdata.org/gbd-results/.

In [2]:
import xarray as xr
import numpy as np
import pandas as pd

In [3]:
# === Path config ===
BMR_DIR = "/glade/work/awells/air_quality/BMR/"

# Reads the CSV file into a DataFrame
# Chronic Obstructive Pulmonary Disease baseline mortality rate by country
df = pd.read_csv(f"{BMR_DIR}IHME-GBD_2021_DATA-00e4ce27-1.csv")

In [4]:
# Options are Number, Percent or Rate (Rate is in number per 100,000)
df = df[df["metric_name"] == "Rate"]

In [5]:
# Calculate the mean across 1990-2009 for each country
df_mean = df.groupby("location_name").mean("year").reset_index()

In [6]:
country = df_mean["location_name"]
val = df_mean["val"]  # the mean value [GBD Results Tool User Guide]
upper = df_mean["upper"]  # WAITING TO CONFIRM UNCERTAINTY RANGE
lower = df_mean["lower"]  # WAITING TO CONFIRM UNCERTAINTY RANGE

In [7]:
data = np.stack([lower, val, upper], axis=1)  # shape (204, 3)

# Create the xarray DataArray
da = xr.DataArray(
    data,
    dims=["country", "quantile"],
    coords={
        "country": country,
        "quantile": ["lower", "mean", "upper"]
    },
    name="BMR_by_country"
)

In [8]:
da.to_netcdf(f"{BMR_DIR}GBD_BMR_Country_COPD_1990-2009.nc")